In [2]:
!pip install transformers pillow torch torchvision

   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.3 MB 262.6 kB/s eta 0:00:17
   ---------------------------------------- 0.0/4.3 MB 245.8 kB/s eta 0:00:18
    --------------------------------------- 0.1/4.3 MB 476.3 kB/s eta 0:00:09
    --------------------------------------- 0.1/4.3 MB 476.3 kB/s eta 0:00:09
    --------------------------------------- 0.1/4.3 MB 476.3 kB/s eta 0:00:09
   - -------------------------------------- 0.1/4.3 MB 448.2 kB/s eta 0:00:10
   - -------------------------------------- 0.2/4.3 MB 565.6 kB/s eta 0:00:08
   - -----------------------------------


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Step 0: Setup - Load environment and dependencies
import os
from dotenv import load_dotenv
load_dotenv()

# Text Embeddings (using sentence-transformers)
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
    
    def generate_embeddings(self, texts):
        return self.model.encode(texts)

embedding_manager = EmbeddingManager()

# Simple Retriever (using ChromaDB from your existing setup)
import chromadb

class SimpleRetriever:
    def __init__(self, embedding_manager):
        self.client = chromadb.PersistentClient(path="C:\\Users\\USER\\Desktop\\pinecone-demo\\data\\vectorstore")
        self.collection = self.client.get_or_create_collection("documents")
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query, top_k=3):
        query_embedding = self.embedding_manager.generate_embeddings([query])[0].tolist()
        results = self.collection.query(query_embeddings=[query_embedding], n_results=top_k)
        return [{"document": doc} for doc in results["documents"][0]] if results["documents"] else []

retriever = SimpleRetriever(embedding_manager)

# LLM Setup (OpenAI)
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

print("✅ Setup complete: embedding_manager, retriever, llm ready!")

✅ Setup complete: embedding_manager, retriever, llm ready!


In [6]:
"""
TASK 3: Vision-RAG System (VLM + RAG)
Goal: Answer questions about images AND documents together
"""

# Cell 1: Install VLM dependencies


# Cell 2: Load CLIP for image-text matching
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Cell 3: Create multimodal embedding function
class MultimodalEmbedding:
    def __init__(self, clip_model, clip_processor, text_embedder):
        self.clip_model = clip_model
        self.clip_processor = clip_processor
        self.text_embedder = text_embedder
    
    def embed_image(self, image_path):
        """Get CLIP embedding for image"""
        image = Image.open(image_path)
        inputs = self.clip_processor(images=image, return_tensors="pt")
        with torch.no_grad():
            image_features = self.clip_model.get_image_features(**inputs)
        return image_features.numpy()[0]
    
    def embed_text(self, text):
        """Get text embedding"""
        return self.text_embedder.generate_embeddings([text])[0]
    
    def similarity(self, image_path, text):
        """Calculate image-text similarity"""
        image = Image.open(image_path)
        inputs = self.clip_processor(
            text=[text], 
            images=image, 
            return_tensors="pt", 
            padding=True
        )
        with torch.no_grad():
            outputs = self.clip_model(**inputs)
            logits_per_image = outputs.logits_per_image
            probs = logits_per_image.softmax(dim=1)
        return probs[0][0].item()

multimodal = MultimodalEmbedding(clip_model, clip_processor, embedding_manager)

# Cell 4: Build Vision-RAG system
class VisionRAG:
    def __init__(self, text_retriever, multimodal_embedder, llm):
        self.text_retriever = text_retriever
        self.multimodal = multimodal_embedder
        self.llm = llm
    
    def answer_with_image(self, question, image_path=None):
        """Answer question using both text docs and images"""
        
        # 1. Retrieve text context
        text_results = self.text_retriever.retrieve(question, top_k=3)
        text_context = "\n\n".join([r['document'] for r in text_results])
        
        # 2. Add image context if provided
        image_context = ""
        if image_path:
            similarity = self.multimodal.similarity(image_path, question)
            image_context = f"\n\nImage relevance score: {similarity:.2f}"
            if similarity > 0.25:
                image_context += "\nThe image is relevant to the question."
        
        # 3. Generate answer
        prompt = f"""Use the following context to answer the question:

Text Context:
{text_context}
{image_context}

Question: {question}

Answer:"""
        
        response = self.llm.invoke([prompt])
        return response.content

vision_rag = VisionRAG(retriever, multimodal, llm)

# Cell 5: Test Vision-RAG
# Add a security architecture diagram image
test_question = "Explain the RAG architecture shown in the diagram"
# answer = vision_rag.answer_with_image(test_question, "path/to/diagram.png")
answer = vision_rag.answer_with_image(test_question)
print(answer)

The RAG architecture shown in the diagram consists of three main components: the Reader, the Analyzer, and the Generator. 

1. Reader: This component is responsible for reading and inputting the raw data or information into the system. It collects data from various sources and prepares it for analysis.

2. Analyzer: The Analyzer component processes the data received from the Reader. It analyzes the data using various algorithms, models, and techniques to extract meaningful insights and patterns. This component is crucial for making sense of the data and identifying trends.

3. Generator: The Generator component takes the analyzed data and generates outputs or reports based on the findings. It can create visualizations, reports, or any other form of output that can help stakeholders understand the insights gained from the data analysis.

Overall, the RAG architecture is designed to efficiently process raw data, analyze it, and generate valuable insights that can be used for decision-mak